# Z_2_4_Using_API_to_add_Headquater_State

**Objective:**
This notebook enhances the existing SEO dataset by mapping issuer CIKs to their headquarters’ state or country using the SEC EDGAR API. This additional location-based information can be used for regional analysis, regulatory categorization, or feature engineering in predictive models.

**Key Tasks:**

- Merge issuer CUSIP8s with a CIK mapping file

- Standardize and clean CIKs to 10-digit format

- Query the SEC EDGAR API to retrieve business address data

- Extract the stateOrCountry field for each unique CIK

- Map the location back to the full dataset

- Export the enriched dataset in both .csv and .parquet formats


In [1]:
# ===========================================
# 1. Load Main SEO Dataset
# ===========================================
import pandas as pd
import requests
import time

file_path = r"data\Updated SEO ISSUES Dataset\SEO_ISSUES_After_ratios_and_pe.parquet"

# Read the SEO dataset enriched with financial ratios and EPS
df = pd.read_parquet(file_path)


In [2]:
# ===========================================
# 2. Load and Merge CIK-CUSIP Mapping
# ===========================================

# Load CIK-CUSIP mapping CSV
mapping_df = pd.read_csv("data/cik-cusip-maps.csv", dtype=str)

# Convert CIK to float to handle existing decimals
mapping_df['cik'] = mapping_df['cik'].astype(float)

# Standardize CUSIP8 format in the SEO dataset
df['Issuer_CUSIP8'] = df['Issuer_CUSIP8'].astype(str).str.upper().str.strip()

# Drop duplicates on cusip8 to ensure clean merge
mapping_df_deduped = mapping_df.drop_duplicates(subset='cusip8')

# Merge CIK into the main dataframe using CUSIP8
df = df.merge(mapping_df_deduped, left_on='Issuer_CUSIP8', right_on='cusip8', how='left')

# Fill missing CIK values with the ones retrieved from the mapping
df['cik_x'] = df['cik_x'].fillna(df['cik_y'])

# Clean up columns
df.drop(columns=['cik_y', 'cusip6', 'cusip8'], inplace=True)
df.rename(columns={'cik_x': 'cik'}, inplace=True)


In [3]:
# ===========================================
# 3. Clean and Standardize CIK Format
# ===========================================

# Function to clean CIK (remove '.0' and pad to 10 digits)
def clean_cik(cik):
    if pd.isna(cik):
        return None
    return str(cik).split('.')[0].zfill(10)

# Apply the cleaning function
df['cik'] = df['cik'].apply(clean_cik)


In [4]:
# ===========================================
# 4. Query SEC EDGAR API to Get Headquarters State/Country
# ===========================================

# Function to retrieve company info from EDGAR
def get_company_info_by_cik(cik):
    if cik is None:
        return None
    url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    headers = {
        "User-Agent": "Supachai Jenpraditwong (sjen591@aucklanduni.ac.nz)"
    }
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"Error retrieving data for CIK {cik}: {e}")
        return None

# Extract the state/country from EDGAR JSON response
def extract_state_from_cik(cik):
    start_time = time.time()
    data = get_company_info_by_cik(cik)
    print(f"Retrieved CIK {cik} in {time.time() - start_time:.2f}s")
    if data:
        addr = data.get('addresses', {}).get('business', {})
        return addr.get('stateOrCountry')
    return None


In [5]:
# ===========================================
# 5. Retrieve State/Country for Unique CIKs
# ===========================================

# Create cleaned version of CIKs for mapping
df['cleaned_cik'] = df['cik'].apply(clean_cik)

# Extract unique valid CIKs
unique_ciks = df['cleaned_cik'].dropna().unique()

# Create a dictionary mapping CIK → state/country (via SEC API)
cik_to_state = {cik: extract_state_from_cik(cik) for cik in unique_ciks}

# Map back to full dataset
df['HQ_State'] = df['cleaned_cik'].map(cik_to_state)


Retrieved CIK 0001160308 in 6.31s
Retrieved CIK 0000944739 in 0.45s
Retrieved CIK 0001080360 in 0.31s
Retrieved CIK 0001319869 in 0.33s
Retrieved CIK 0001319009 in 0.37s
Retrieved CIK 0001161154 in 0.38s
Retrieved CIK 0001171014 in 0.38s
Retrieved CIK 0001368993 in 0.51s
Retrieved CIK 0000915840 in 0.32s
Retrieved CIK 0001376339 in 0.33s
Retrieved CIK 0001129425 in 0.37s
Retrieved CIK 0001061219 in 0.37s
Retrieved CIK 0001179060 in 0.43s
Retrieved CIK 0001157601 in 0.39s
Retrieved CIK 0001350684 in 0.37s
Retrieved CIK 0000926617 in 0.33s
Retrieved CIK 0001011509 in 0.37s
Retrieved CIK 0001171298 in 0.40s
Retrieved CIK 0000912263 in 0.37s
Retrieved CIK 0001437071 in 0.39s
Retrieved CIK 0000895051 in 0.40s
Retrieved CIK 0001130166 in 0.42s
Retrieved CIK 0000007789 in 0.39s
Retrieved CIK 0001319229 in 0.51s
Retrieved CIK 0000911635 in 0.64s
Retrieved CIK 0001358831 in 0.45s
Retrieved CIK 0000086115 in 0.41s
Retrieved CIK 0000702325 in 0.63s
Retrieved CIK 0000934473 in 0.36s
Retrieved CIK 

In [6]:
# ===========================================
# 6. Final Clean-up and Export
# ===========================================

# Fill missing state values with HQ_State if available
df['state'] = df['state'].fillna(df['HQ_State'])

# Drop temporary column
df.drop(columns='HQ_State', inplace=True)

# Export enriched dataset with issuer HQ state/country
df.to_csv(r"data\Updated SEO ISSUES Dataset\SEO_ISSUES_After_ratios_and_pe_state.csv", index=False)
df.to_parquet(r"data\Updated SEO ISSUES Dataset\SEO_ISSUES_After_ratios_and_pe_state.parquet", index=False)
